# Separate human and mouse protein tables without data loss


This migrates the 0.46.0 combined host table. Species come from declared source provenance. Identity-only rows use the local UniProt mappings and human bridge sources; every source hash is recorded. The migration refuses conflicts, unknown columns and unassigned rows.

The source is retrieved from its recorded Git revision if the legacy cache has been removed. The two resulting tables retain UniProt keys. Full gene spaces and their layouts are a later work package. A rerun refuses to overwrite outputs that have acquired further data.

In [1]:
from pathlib import Path
import subprocess, tempfile
from scripts.split_host_tables import migrate
from starplast import paths
data = Path('starplast/data')
temporary = tempfile.TemporaryDirectory()
source = data / 'host_proteins.parquet'
if not source.exists():
    source = Path(temporary.name) / 'host_proteins.parquet'
    source.write_bytes(subprocess.check_output(['git', 'show', 'f02d075:starplast/data/host_proteins.parquet']))
report = migrate(source, data, paths.dataset_root(), write=True)
temporary.cleanup()
{code: (record['rows'], len(record['coverage'])) for code, record in report['tables'].items()}

{'Hs': (20989, 18), 'Mm': (15590, 7)}

The migration checks identifier partitioning, every per-species cell against the original, per-column coverage, and the parquet round trip before completing. Sources and coverage counts are stored in host_species_migration.json.

In [2]:
print('All identifiers retained:', report['rows_before'] == sum(t['rows'] for t in report['tables'].values()))
print('Identity-only row memberships:', {c: len(set().union(*(set(r['identity_only_rows']) for r in report['identity_sources'] if r['organism'] == c))) for c in report['tables']})

All identifiers retained: True
Identity-only row memberships: {'Hs': 49, 'Mm': 5}
